# Huấn luyện mô hình gốc DeepVQE với bộ VCTK-DEMAND (VoiceBank-DEMAND)
Notebook này thiết lập môi trường để tải bộ dữ liệu, liên kết Google Drive, clone mã nguồn DeepVQE, xử lý metadata (file csv) và huấn luyện mô hình.

## 1. Cài đặt môi trường & Kết nối Google Drive

In [ ]:
!pip install torchaudio speechbrain tensorboard soundfile pandas tqdm einops

from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/DeepVQE_Workspace'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f"Thư mục làm việc hiện tại: {os.getcwd()}")

## 2. Clone mã nguồn DeepVQE

In [ ]:
# Sử dụng thư viện gốc của tác giả
GIT_REPO = "https://github.com/Xiaobin-Rong/deepvqe.git"

if not os.path.exists("deepvqe"):
    !git clone {GIT_REPO}
else:
    print("Thư mục deepvqe đã tồn tại.")

os.chdir("deepvqe")
print(f"Đã vào thư mục mã nguồn: {os.getcwd()}")

## 3. Tải bộ dữ liệu VoiceBank-DEMAND
Dữ liệu gốc được tải từ Đại học Edinburgh (chiếm khoảng 4GB - 5GB sau khi giải nén).

**Lưu ý**: VCTK-DEMAND gốc có sample rate 48kHz. Code training sẽ tự resample xuống 16kHz.

In [ ]:
import os
import zipfile

data_dir = "/content/drive/MyDrive/DeepVQE_Workspace/data/voicebank-demand"
os.makedirs(data_dir, exist_ok=True)

datasets = {
    "clean_trainset_28spk_wav": "https://datashare.ed.ac.uk/bitstream/handle/10283/2791/clean_trainset_28spk_wav.zip",
    "noisy_trainset_28spk_wav": "https://datashare.ed.ac.uk/bitstream/handle/10283/2791/noisy_trainset_28spk_wav.zip",
    "clean_testset_wav": "https://datashare.ed.ac.uk/bitstream/handle/10283/2791/clean_testset_wav.zip",
    "noisy_testset_wav": "https://datashare.ed.ac.uk/bitstream/handle/10283/2791/noisy_testset_wav.zip"
}

for folder_name, url in datasets.items():
    extract_path = os.path.join(data_dir, folder_name)
    zip_path = os.path.join(data_dir, folder_name + ".zip")
    
    # Kiểm tra xem thư mục giải nén đã có dữ liệu chưa
    if os.path.exists(extract_path) and len(os.listdir(extract_path)) > 0:
        print(f"Thư mục {folder_name} đã tồn tại và có dữ liệu trong Drive, bỏ qua tải.")
    else:
        if not os.path.exists(zip_path):
            print(f"Đang tải {folder_name}.zip về Drive...")
            !wget -nc -P {data_dir} {url}
        else:
            print(f"File {folder_name}.zip đã tồn tại trong Drive, tiến hành giải nén.")
        
        print(f"Đang giải nén {folder_name}.zip...")
        try:
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(data_dir)
            # (Tùy chọn) Xóa file zip để tiết kiệm dung lượng Drive
            # os.remove(zip_path)
            print(f"Đã giải nén xong {folder_name}.zip")
        except zipfile.BadZipFile:
            print(f"Lỗi: File {folder_name}.zip bị hỏng. Vui lòng xóa file này và chạy lại ô này.")

print("\nKiểm tra và chuẩn bị dữ liệu VCTK-DEMAND hoàn tất!")

## 4. Xử lý và phân chia Dataset (Tạo file CSV)
Để framework (ví dụ như SpeechBrain) dễ dàng tải dữ liệu, ta sẽ tạo các file metadata dạng CSV (`train.csv`, `valid.csv`, `test.csv`).

In [ ]:
import glob
import pandas as pd

def create_csv(clean_dir, noisy_dir, output_csv):
    clean_files = sorted(glob.glob(os.path.join(clean_dir, "*.wav")))
    noisy_files = sorted(glob.glob(os.path.join(noisy_dir, "*.wav")))
    
    assert len(clean_files) == len(noisy_files), f"Số lượng file không khớp! ({len(clean_files)} != {len(noisy_files)})"
    
    data = []
    for c, n in zip(clean_files, noisy_files):
        filename = os.path.basename(c)
        data.append({
            "ID": filename.replace(".wav", ""),
            "clean_wav": os.path.abspath(c),
            "noisy_wav": os.path.abspath(n)
        })
        
    df = pd.DataFrame(data)
    df.to_csv(output_csv, index=False)
    print(f"Đã tạo {output_csv} với {len(df)} mẫu.")

base_dir = data_dir
# Tạo file CSV thô
create_csv(f"{base_dir}/clean_trainset_28spk_wav", f"{base_dir}/noisy_trainset_28spk_wav", f"{base_dir}/train_full.csv")
create_csv(f"{base_dir}/clean_testset_wav", f"{base_dir}/noisy_testset_wav", f"{base_dir}/test.csv")

# Tách train_full thành train (90%) và valid (10%)
df_train_full = pd.read_csv(f"{base_dir}/train_full.csv")
df_train_full = df_train_full.sample(frac=1, random_state=42).reset_index(drop=True)

split_idx = int(len(df_train_full) * 0.90)
df_train_full.iloc[:split_idx].to_csv(f"{base_dir}/train.csv", index=False)
df_train_full.iloc[split_idx:].to_csv(f"{base_dir}/valid.csv", index=False)

print("Hoàn tất tạo metadata (train.csv, valid.csv, test.csv)!")

## 5. Cấu hình Train (hparams.yaml)
File cấu hình hyperparameters cho DeepVQE.

**Lưu ý**: Đã bỏ `optimizer` khỏi `recoverables` vì `opt_class` là class chưa khởi tạo — SpeechBrain Brain sẽ tự tạo và quản lý optimizer instance.

In [ ]:
yaml_content = """
seed: 1234
__set_seed: !apply:torch.manual_seed [!ref <seed>]

# Data paths
data_folder: /content/drive/MyDrive/DeepVQE_Workspace/data/voicebank-demand
train_annotation: !ref <data_folder>/train.csv
valid_annotation: !ref <data_folder>/valid.csv
test_annotation: !ref <data_folder>/test.csv

# Audio parameters
sample_rate: 16000
n_fft: 512
hop_length: 256
win_length: 512

# Training parameters
batch_size: 4
N_epochs: 100
lr: 0.0002
grad_clip: 5.0

# Loss weights
loss_waveform_weight: 0.5
loss_spectral_weight: 0.5

# Dataloaders
dataloader_options:
    batch_size: !ref <batch_size>
    num_workers: 2
    drop_last: False

# Model Definition
model: !new:deepvqe.DeepVQE

# Optimizer
opt_class: !name:torch.optim.Adam
    lr: !ref <lr>

epoch_counter: !new:speechbrain.utils.epoch_loop.EpochCounter
    limit: !ref <N_epochs>

# Checkpointer - lưu lại mô hình trực tiếp lên Google Drive
# Lưu ý: KHÔNG lưu optimizer ở đây vì opt_class là class, không phải instance.
# SpeechBrain Brain.fit() tự quản lý optimizer.
output_folder: /content/drive/MyDrive/DeepVQE_Workspace/checkpoints/deepvqe_vctk/<seed>
save_folder: !ref <output_folder>/save
checkpointer: !new:speechbrain.utils.checkpoints.Checkpointer
    checkpoints_dir: !ref <save_folder>
    recoverables:
        model: !ref <model>
        counter: !ref <epoch_counter>
"""

with open("hparams_vctk.yaml", "w") as f:
    f.write(yaml_content)
print("Đã lưu cấu hình vào hparams_vctk.yaml")

## 6. Code Train
Training loop sử dụng SpeechBrain `Brain` class.

**Các sửa đổi so với phiên bản cũ:**
- ✅ Thêm resample 16kHz trong `audio_pipeline` (VCTK-DEMAND gốc là 48kHz)
- ✅ Tạo `hann_window` một lần trong `on_stage_start` thay vì mỗi forward call
- ✅ Sửa length mismatch giữa predictions và clean_wavs
- ✅ Thêm spectral loss (STFT magnitude L1) bên cạnh waveform L1
- ✅ Thêm gradient clipping cho training ổn định
- ✅ Sửa lỗi checkpointer (bỏ optimizer khỏi recoverables)

In [ ]:
import os
import sys
import torch
import torchaudio
from speechbrain.core import Brain
from speechbrain.dataio.dataset import DynamicItemDataset
from hyperpyyaml import load_hyperpyyaml

# Đảm bảo đang đứng đúng thư mục chứa mã nguồn
work_dir = '/content/drive/MyDrive/DeepVQE_Workspace/deepvqe'
if os.path.exists(work_dir):
    os.chdir(work_dir)
    if work_dir not in sys.path:
        sys.path.insert(0, work_dir)

# Đọc cấu hình từ file YAML
with open('hparams_vctk.yaml') as fin:
    hparams = load_hyperpyyaml(fin)

# ============================================================
# 1. Chuẩn bị DataLoader (SpeechBrain DynamicItemDataset)
# ============================================================

TARGET_SR = hparams.get('sample_rate', 16000)

def audio_pipeline(file_path):
    """Đọc file audio và resample về TARGET_SR nếu cần."""
    sig, sr = torchaudio.load(file_path)
    if sr != TARGET_SR:
        sig = torchaudio.functional.resample(sig, sr, TARGET_SR)
    return sig.squeeze(0)  # [1, T] -> [T] để SpeechBrain tự padding

train_set = DynamicItemDataset.from_csv(hparams['train_annotation'])
valid_set = DynamicItemDataset.from_csv(hparams['valid_annotation'])

for dataset in [train_set, valid_set]:
    dataset.add_dynamic_item(audio_pipeline, takes="noisy_wav", provides="noisy_sig")
    dataset.add_dynamic_item(audio_pipeline, takes="clean_wav", provides="clean_sig")
    dataset.set_output_keys(["id", "noisy_sig", "clean_sig"])

# ============================================================
# 2. Định nghĩa Recipe Train (kế thừa từ speechbrain.Brain)
# ============================================================

class VQE_Brain(Brain):
    
    def on_stage_start(self, stage, epoch=None):
        """Tạo hann_window MỘT LẦN cho mỗi stage, tránh tạo lại trong mỗi forward."""
        super().on_stage_start(stage, epoch)
        
        # Sử dụng getattr với giá trị mặc định để tránh lỗi nếu quên chạy lại cell 5
        win_length = getattr(self.hparams, 'win_length', 512)
        self.window = torch.hann_window(win_length).to(self.device)
        self.n_fft = getattr(self.hparams, 'n_fft', 512)
        self.hop_length = getattr(self.hparams, 'hop_length', 256)
        
        # Khởi tạo tracking loss cho validation
        if stage != "train":
            self.valid_loss_sum = 0.0
            self.valid_loss_count = 0

    def compute_forward(self, batch, stage):
        noisy_wavs, lens = batch.noisy_sig
        noisy_wavs = noisy_wavs.to(self.device)
        
        # STFT: (B, T) -> (B, F, T_frames, 2)
        noisy_stft = torch.stft(
            noisy_wavs, 
            n_fft=self.n_fft, 
            hop_length=self.hop_length, 
            win_length=self.n_fft,
            window=self.window, 
            return_complex=True
        )
        noisy_stft_real = torch.view_as_real(noisy_stft)  # (B, F, T_frames, 2)
        
        # DeepVQE: (B, F, T_frames, 2) -> (B, F, T_frames, 2)
        pred_stft_real = self.modules.model(noisy_stft_real)
        
        # ISTFT: khôi phục waveform, cắt về đúng length ban đầu
        pred_stft_complex = torch.complex(pred_stft_real[..., 0], pred_stft_real[..., 1])
        enhanced_wavs = torch.istft(
            pred_stft_complex, 
            n_fft=self.n_fft, 
            hop_length=self.hop_length, 
            win_length=self.n_fft,
            window=self.window, 
            length=noisy_wavs.shape[1]
        )
        
        # Trả về cả waveform và STFT để tính multi-domain loss
        return enhanced_wavs, pred_stft_real

    def compute_objectives(self, predictions, batch, stage):
        enhanced_wavs, pred_stft = predictions
        clean_wavs, lens = batch.clean_sig
        clean_wavs = clean_wavs.to(self.device)
        
        # === Fix length mismatch ===
        # Sau STFT->model->ISTFT, enhanced_wavs có thể khác length với clean_wavs
        min_len = min(enhanced_wavs.shape[-1], clean_wavs.shape[-1])
        enhanced_wavs = enhanced_wavs[..., :min_len]
        clean_wavs = clean_wavs[..., :min_len]
        
        # --- Loss 1: Waveform L1 ---
        loss_wav = torch.nn.functional.l1_loss(enhanced_wavs, clean_wavs)
        
        # --- Loss 2: Spectral Magnitude L1 (STFT domain) ---
        # Tính STFT của clean signal để so sánh trên spectral domain
        clean_stft = torch.stft(
            clean_wavs, 
            n_fft=self.n_fft, 
            hop_length=self.hop_length, 
            win_length=self.n_fft,
            window=self.window, 
            return_complex=True
        )
        clean_stft_real = torch.view_as_real(clean_stft)  # (B, F, T_frames, 2)
        
        # Đảm bảo T_frames khớp giữa pred và clean
        min_t = min(pred_stft.shape[2], clean_stft_real.shape[2])
        pred_stft_t = pred_stft[:, :, :min_t, :]
        clean_stft_t = clean_stft_real[:, :, :min_t, :]
        
        # L1 trên magnitude spectrum
        pred_mag = torch.sqrt(pred_stft_t[..., 0]**2 + pred_stft_t[..., 1]**2 + 1e-12)
        clean_mag = torch.sqrt(clean_stft_t[..., 0]**2 + clean_stft_t[..., 1]**2 + 1e-12)
        loss_spec = torch.nn.functional.l1_loss(pred_mag, clean_mag)
        
        # --- Combined Loss ---
        # Sử dụng getattr để lấy giá trị mặc định, tránh lỗi cũ
        w_wav = getattr(self.hparams, 'loss_waveform_weight', 0.5)
        w_spec = getattr(self.hparams, 'loss_spectral_weight', 0.5)
        loss = w_wav * loss_wav + w_spec * loss_spec
        
        # Track validation loss
        if stage != "train":
            self.valid_loss_sum += loss.item()
            self.valid_loss_count += 1
        
        return loss
    
    def on_stage_end(self, stage, stage_loss, epoch=None):
        """Log kết quả."""
        super().on_stage_end(stage, stage_loss, epoch)
        if stage == "valid":
            avg_valid_loss = self.valid_loss_sum / max(self.valid_loss_count, 1)
            print(f"  Epoch {epoch} | Valid Loss: {avg_valid_loss:.6f}")

# ============================================================
# 3. Khởi tạo và Train
# ============================================================

vqe_brain = VQE_Brain(
    modules={'model': hparams['model']},
    opt_class=hparams['opt_class'],
    hparams=hparams,
    checkpointer=hparams['checkpointer'], 
)

# Chạy Train Loop (tự động Resume từ checkpoint nếu có)
vqe_brain.fit(
    vqe_brain.hparams.epoch_counter,
    train_set,
    valid_set,
    train_loader_kwargs=hparams.get('dataloader_options', {'batch_size': 4}),
    valid_loader_kwargs=hparams.get('dataloader_options', {'batch_size': 4}),
)
print("\n=== Huấn luyện hoàn tất! ===")
print(f"Checkpoint đã được lưu tại: {hparams.get('save_folder', 'save')}")

## 7. Kiểm tra nhanh sau training
Chạy inference trên một mẫu test để xác nhận model hoạt động.

In [ ]:
import torchaudio
import IPython.display as ipd

# Load một file test mẫu
test_csv = hparams['test_annotation']
import pandas as pd
df_test = pd.read_csv(test_csv)
sample = df_test.iloc[0]

noisy_path = sample['noisy_wav']
clean_path = sample['clean_wav']

sig_noisy, sr = torchaudio.load(noisy_path)
if sr != TARGET_SR:
    sig_noisy = torchaudio.functional.resample(sig_noisy, sr, TARGET_SR)

# Inference
model = hparams['model'].eval().to('cuda' if torch.cuda.is_available() else 'cpu')
device = next(model.parameters()).device
window = torch.hann_window(hparams.get('n_fft', 512)).to(device)

with torch.no_grad():
    noisy_wav = sig_noisy.squeeze(0).to(device)
    stft = torch.stft(noisy_wav, n_fft=512, hop_length=256, win_length=512, 
                      window=window, return_complex=True)
    stft_real = torch.view_as_real(stft).unsqueeze(0)  # (1, F, T, 2)
    pred_stft = model(stft_real)
    pred_complex = torch.complex(pred_stft[..., 0], pred_stft[..., 1]).squeeze(0)
    enhanced = torch.istft(pred_complex, n_fft=512, hop_length=256, win_length=512,
                           window=window, length=noisy_wav.shape[0])

print(f"Sample: {sample['ID']}")
print(f"Noisy:")
ipd.display(ipd.Audio(sig_noisy.squeeze().cpu().numpy(), rate=TARGET_SR))
print(f"Enhanced:")
ipd.display(ipd.Audio(enhanced.cpu().numpy(), rate=TARGET_SR))

sig_clean, sr_c = torchaudio.load(clean_path)
if sr_c != TARGET_SR:
    sig_clean = torchaudio.functional.resample(sig_clean, sr_c, TARGET_SR)
print(f"Clean (ground truth):")
ipd.display(ipd.Audio(sig_clean.squeeze().cpu().numpy(), rate=TARGET_SR))